In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from netgen.occ import *

In [ ]:
def create_mesh(L=2.2, H=0.41, r=0.05, maxh=0.05):
    shape = Rectangle(L,H).Circle(0.2,0.19,r).Reverse().Face()
    shape.edges.name="wall"
    shape.edges.Min(X).name="inlet"
    shape.edges.Max(X).name="outlet"
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=maxh)).Curve(3)
    return mesh

In [ ]:
def get_gfu(mesh, nu, U_m, H=0.41):
    V = VectorH1(mesh,order=3, dirichlet="wall|inlet")
    Q = H1(mesh,order=2)
    X = V*Q
    u,p = X.TrialFunction()
    v,q = X.TestFunction()

    stokes = (
        nu * InnerProduct(grad(u), grad(v))
        + div(u) * q 
        + div(v) * p 
        - 10 ** (-10) * p * q
        ) * dx

    a = BilinearForm(stokes).Assemble()
    f = LinearForm(X).Assemble()
    gfu = GridFunction(X)

    # set inlet velocity
    uin = CoefficientFunction((U_m, 0))

    gfu.components[0].Set(uin, definedon=mesh.Boundaries("inlet"))
    inv_stokes = a.mat.Inverse(X.FreeDofs())
    res = f.vec - a.mat*gfu.vec

    params = (V, X, v, u, stokes, a)
    
    return gfu, params

In [ ]:
def reynolds_number(U_m, nu):
    L=0.1
    return U_m*L/nu

In [ ]:
def strouhal_number(N, t, U_m):
    L = 0.1
    f = N / t
    return f * L / U_m

### Re = 50

In [ ]:
nu = 0.005
U_m = 2.5
delta_t = 0.0001
t_end = 2.0

# Result of manual counting of vortex shedding in the animation
N = 0.0
St = strouhal_number(N, t_end, U_m)
Re = reynolds_number(U_m, nu)

print(f"Reynolds number: {Re}")
print(f"Strouhal number: {St}")

### Re = 100

In [ ]:
nu = 0.005
U_m = 5.0
delta_t = 0.0001
t_end = 2.0

# Result of manual counting of vortex shedding in the animation
N = 26.5
St = strouhal_number(N, t_end, U_m)
Re = reynolds_number(U_m, nu)

print(f"Reynolds number: {Re}")
print(f"Strouhal number: {St}")

### Re = 150

In [ ]:
nu = 0.005
U_m = 7.5
delta_t = 0.0001
t_end = 2.0

# Result of manual counting of vortex shedding in the animation
N = 42.0
St = strouhal_number(N, t_end, U_m)
Re = reynolds_number(U_m, nu)

print(f"Reynolds number: {Re}")
print(f"Strouhal number: {St}")

### Re = 250

In [ ]:
nu = 0.005
U_m = 12.5
delta_t = 0.00004
t_end = 1.0

# Result of manual counting of vortex shedding in the animation
N = 34.5
St = strouhal_number(N, t_end, U_m)
Re = reynolds_number(U_m, nu)

print(f"Reynolds number: {Re}")
print(f"Strouhal number: {St}")

### Re = 350

In [ ]:
nu = 0.005
U_m = 17.5
delta_t = 0.00002
t_end = 0.5

# Result of manual counting of vortex shedding in the animation
N = 24.5
St = strouhal_number(N, t_end, U_m)
Re = reynolds_number(U_m, nu)

print(f"Reynolds number: {Re}")
print(f"Strouhal number: {St}")

### Re = 500

In [ ]:
nu = 0.005
U_m = 25.0
delta_t = 0.00001
t_end = 0.25

# Result of manual counting of vortex shedding in the animation
N = 17.0
St = strouhal_number(N, t_end, U_m)
Re = reynolds_number(U_m, nu)

print(f"Reynolds number: {Re}")
print(f"Strouhal number: {St}")

Run simulation for given parameters

In [ ]:
nu = 0.005
U_m = 7.5
delta_t = 0.0001
t_end = 2.0

mesh = create_mesh()
gfu, params = get_gfu(mesh, nu, U_m)

Re = reynolds_number(U_m, nu)
print(f"Reynolds number: {Re}")

#######

V, X, v, u, stokes, a = params
mstar = BilinearForm(u*v*dx+delta_t*stokes).Assemble()
inv = mstar.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky")

conv = BilinearForm(X, nonassemble=True)
conv += (Grad(u) * u) * v * dx

t = 0; i = 0

gfut = GridFunction(V, multidim=0)
vel = gfu.components[0]

scene = Draw (gfu.components[0], mesh, min=0, max=U_m+5, autoscale=False)
display(scene)
tw = widgets.Text(value='t = 0')
display(tw)

with TaskManager():
    while t < t_end:
        res = conv.Apply(gfu.vec) + a.mat*gfu.vec
        gfu.vec.data -= delta_t * inv * res    

        t = t + delta_t; i = i + 1

        if i%10 == 0: scene.Redraw()
        if i%50 == 0: gfut.AddMultiDimComponent(vel.vec)
        if i%100 == 0: tw.value = "t = " + str(t)
    tw.value = "t = "+str(t_end)


In [ ]:
Draw (gfut, mesh, interpolate_multidim=True, animate=True, min=0, max=U_m, autoscale=False)